In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 138.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 166.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 207.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 175.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

from unsloth import FastLanguageModel
import json
from datasets import Dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
# Load the model
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-8B-Base",
    max_seq_length = 8192,   # longer context to accommodate IR code blocks
    dtype = torch.bfloat16,   # A100/H100 natively supports bfloat16 for better precision
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.3.3: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/6.75G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/qwen3-8b-base-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [8]:
# 2️⃣ Load dataset from HuggingFace
from datasets import load_dataset

ds = load_dataset("YangziResearch/IR-OptSet", split="train")

# --- A100 CONFIGURATION ---
DRY_RUN = False  # Set to False to train on the full 170k dataset!


# Split the dataset based on DRY_RUN mode
if DRY_RUN:
    print("Running in DRY_RUN mode (2200 samples total)...")
    ds = ds.select(range(2200))
    train_test = ds.train_test_split(test_size=200, seed=42)
else:
    print("Running in FULL mode (10k samples)...")
    selected = ds.select(range(20000))
    train_test = selected.train_test_split(test_size=200, seed=42)

train_ds = train_test["train"]
test_ds = train_test["test"]

# 3️⃣ Alpaca-style prompt template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def to_text(batch):
    instruction = "Given unoptimized LLVM IR, produce the equivalent O3-optimized IR."
    texts = []
    for inp, out in zip(batch['preprocessed_ir'], batch['o3_ir']):
        if not isinstance(inp, str): inp = json.dumps(inp, ensure_ascii=False)
        if not isinstance(out, str): out = json.dumps(out, ensure_ascii=False)
        texts.append(alpaca_prompt.format(instruction, inp, out) + EOS_TOKEN)
    return {"text": texts}



# CRITICAL FIX for 170k dataset: batched processing to prevent RAM crash
dataset = train_ds.map(
    to_text,
    batched=True,
    batch_size=1000,
    num_proc=8,
    remove_columns=train_ds.column_names
)


Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/19 [00:00<?, ?it/s]

Running in FULL mode (10k samples)...


Map (num_proc=8):   0%|          | 0/19800 [00:00<?, ? examples/s]

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128,  # A100/H100 has 40GB+ — can handle higher rank for better learning
    target_modules= [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_alpha = 128 * 2,  # keep 2x rank
    lora_dropout = 0.05,  # small dropout to prevent overfitting on 1k samples
    bias = "none",
    use_gradient_checkpointing='unsloth'
)

Unsloth: Already have LoRA adapters! We shall skip this step.


In [10]:

# ── Pre-training length filter ──────────────────────────────────────────────
# SFTTrainer silently truncates examples that exceed max_seq_length.
# Training on truncated (cut-off) IR teaches the model that incomplete IR is
# valid output. We filter them out entirely to keep the training data clean.
MAX_SEQ_LENGTH = 8192
def is_within_length(batch):
    token_counts = [len(tokenizer.encode(t)) for t in batch["text"]]
    return [c <= MAX_SEQ_LENGTH for c in token_counts]
before_count = len(dataset)
dataset = dataset.filter(is_within_length, batched=True, batch_size=64)
after_count = len(dataset)
print(f"Dataset filtered: {before_count} → {after_count} examples "
      f"({before_count - after_count} removed due to length > {MAX_SEQ_LENGTH} tokens)")
# ─────────────────────────────────────────────────────────────────────────────

from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=8192,
    args=SFTConfig(
        per_device_train_batch_size=8,      # H100 natively handles large batches
        gradient_accumulation_steps=8,      # effective batch = 64
        warmup_steps=10 if DRY_RUN else 15,                    # ~10% of 1 epoch (~140 steps for 10k run)
        num_train_epochs=1,                 # 1 pass over the 9k training data
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        logging_steps=1 if DRY_RUN else 10,
        save_strategy='steps',
        save_steps=50,                      # Save more frequently since total steps is ~140
        save_total_limit=2,
        fp16=False,                          # H100 strongly prefers bf16
        bf16=True,                           # H100 natively supports bf16
        output_dir="/content/drive/MyDrive/mini-prot-checkpoints",
        optim='adamw_8bit',
    )
)

Filter:   0%|          | 0/19800 [00:00<?, ? examples/s]

Dataset filtered: 19800 → 10616 examples (9184 removed due to length > 8192 tokens)


Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/10616 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [11]:
trainer.train() # this does the traning

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,616 | Num Epochs = 1 | Total steps = 166
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 8 x 1) = 64
 "-____-"     Trainable parameters = 349,175,808 of 8,539,911,168 (4.09% trained)


Step,Training Loss
10,0.243729
20,0.139066
30,0.101886
40,0.084619
50,0.075709
60,0.069421
70,0.066460
80,0.062722
90,0.062306
100,0.058573


TrainOutput(global_step=166, training_loss=0.0803313124251653, metrics={'train_runtime': 11391.6226, 'train_samples_per_second': 0.932, 'train_steps_per_second': 0.015, 'total_flos': 2.359573544825377e+18, 'train_loss': 0.0803313124251653, 'epoch': 1.0})

In [14]:
model.save_pretrained_gguf(
    "/content/drive/MyDrive/mini-prot-final-model",
    tokenizer,
    quantization_method="f16",
)


Unsloth: Merging model weights to 16-bit format...


config.json: 0.00B [00:00, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:21<00:00, 20.47s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/mini-prot-final-model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...


RuntimeError: Unsloth: GGUF conversion failed: Unsloth: Failed to convert model to GGUF with command `/usr/bin/python3 /root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py --outfile qwen3-8b-base.BF16.gguf --outtype bf16 --split-max-size 50G /content/drive/MyDrive/mini-prot-final-model`: Command '['/usr/bin/python3', '/root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py', '--outfile', 'qwen3-8b-base.BF16.gguf', '--outtype', 'bf16', '--split-max-size', '50G', '/content/drive/MyDrive/mini-prot-final-model']' returned non-zero exit status 1.

In [13]:
model.save_pretrained("/content/drive/MyDrive/mini-prot-lora-model")
tokenizer.save_pretrained("/content/drive/MyDrive/mini-prot-lora-model")

('/content/drive/MyDrive/mini-prot-lora-model/tokenizer_config.json',
 '/content/drive/MyDrive/mini-prot-lora-model/tokenizer.json')

In [ ]:
!apt-get update && apt-get install -y llvm


Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,389 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,821 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,780 kB]
Get:12 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/p

In [ ]:
%%writefile mca_sanity_check.py
import subprocess
import re
import sys
import os

def get_mca_cycles(filepath):
    chk = subprocess.run(["opt", "-opaque-pointers", "--passes=verify", "-disable-output", filepath], capture_output=True)
    if chk.returncode != 0: return f"Syntax_Error: {chk.stderr.decode()[:100].strip()}"

    asm = filepath + ".s"
    clean_asm = filepath + "_clean.s"
    try:
        subprocess.run(["llc", "-opaque-pointers", "-O0", filepath, "-o", asm], check=True, capture_output=True)
        with open(asm, 'r') as f_in, open(clean_asm, 'w') as f_out:
            for line in f_in:
                if not line.strip().startswith('.'): f_out.write(line)
        mca_result = subprocess.run(["llvm-mca", "-mcpu=skylake", clean_asm], capture_output=True, text=True, check=True)
        cycles_match = re.search(r"Total Cycles:\s+(\d+)", mca_result.stdout)

        if os.path.exists(asm): os.remove(asm)
        if os.path.exists(clean_asm): os.remove(clean_asm)

        if cycles_match: return int(cycles_match.group(1))
        return "MCA_Failed"
    except Exception:
        return "MCA_Error"

def evaluate(pre_ir_path, llvm_ir_path, llm_ir_path):
    pre_cycles = get_mca_cycles(pre_ir_path)
    llvm_cycles = get_mca_cycles(llvm_ir_path)
    llm_cycles = get_mca_cycles(llm_ir_path)

    imp_pre_vs_llm = "N/A"
    if isinstance(pre_cycles, int) and isinstance(llm_cycles, int):
        imp_pre_vs_llm = f"{(((pre_cycles - llm_cycles)/max(pre_cycles, 1))*100):.2f}%"

    imp_llvm_vs_llm = "N/A"
    if isinstance(llvm_cycles, int) and isinstance(llm_cycles, int):
        imp_llvm_vs_llm = f"{(((llvm_cycles - llm_cycles)/max(llvm_cycles, 1))*100):.2f}%"

    print(f"{pre_cycles},{llvm_cycles},{llm_cycles},{imp_pre_vs_llm},{imp_llvm_vs_llm}")

if __name__ == "__main__":
    evaluate(sys.argv[1], sys.argv[2], sys.argv[3])



Overwriting mca_sanity_check.py


In [ ]:
# Put model in inference mode
FastLanguageModel.for_inference(model)

import os
import subprocess

csv_file = "/content/drive/MyDrive/mca_evaluation_log.csv"
print(f"\n{'='*80}")
print(f"STARTING COMPREHENSIVE EVALUATION ON {len(test_ds)} SAMPLES")
print(f"RESULTS LOGGED TO: {csv_file}")
print(f"{'='*80}\n")

with open(csv_file, 'w') as f:
    f.write("Row_ID,Preprocessed_Cycles,LLVM_O3_Cycles,LLM_O3_Cycles,Improvement(Pre_vs_LLM),Improvement(LLVM_vs_LLM)\n")

print(f"{'Row ID':<8} | {'PRE':<12} | {'LLVM(-O3)':<12} | {'LLM(-O3)':<12} | {'Pre vs LLM':<12} | {'LLVM vs LLM':<12}")
print("-" * 80)

for i, example in enumerate(test_ds):
    pre_ir = example["preprocessed_ir"]
    llvm_ir = example["o3_ir"]

    prompt = alpaca_prompt.format(
        "Given unoptimized LLVM IR, produce the equivalent O3-optimized IR.",
        pre_ir,
        "",
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    if inputs['input_ids'].shape[1] > 24000:
        print(f'Skipping huge {inputs["input_ids"].shape[1]} token file...')
        continue
    outputs = model.generate(**inputs, max_new_tokens=8192, use_cache=True, temperature=0.3, do_sample=True, top_p=0.9)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### Response:\n" in response:
        llm_ir = response.split("### Response:\n")[-1]
    else:
        llm_ir = response

    with open("pre.ll", "w") as f: f.write(pre_ir)
    with open("llvm.ll", "w") as f: f.write(llvm_ir)
    with open("llm.ll", "w") as f: f.write(llm_ir)

    result = subprocess.run(["python", "mca_sanity_check.py", "pre.ll", "llvm.ll", "llm.ll"], capture_output=True, text=True)
    csv_data = result.stdout.strip()

    row_string = f"{i},{csv_data}"
    with open(csv_file, 'a') as f:
        f.write(row_string + "\n")

    cols = row_string.split(",")
    if len(cols) == 6:
        print(f"{cols[0]:<8} | {cols[1]:<12} | {cols[2]:<12} | {cols[3]:<12} | {cols[4]:<12} | {cols[5]:<12}")
    else:
        print(f"{i:<8} | ERROR: {csv_data}")

print(f"\n{'='*80}")
print(f"EVALUATION COMPLETE - Logged entirely to Google Drive.")
print(f"{'='*80}\n")




STARTING COMPREHENSIVE EVALUATION ON 200 SAMPLES
RESULTS LOGGED TO: /content/drive/MyDrive/mca_evaluation_log.csv

Row ID   | PRE          | LLVM(-O3)    | LLM(-O3)     | Pre vs LLM   | LLVM vs LLM 
--------------------------------------------------------------------------------
0        | ERROR: Syntax_Error: invalid behavior operand in module flag (unexpected constant)
i32 8
opt: pre.ll: error: input module,Syntax_Error: opt: llvm.ll:22:24: error: expected ')' at end of argument list
declare void @free(ptr allocptr noca,Syntax_Error: opt: llm.ll:8:3: error: instruction expected to be numbered '%1'
  %0 = load ptr, ptr @_ZL10g_instan,N/A,N/A
Skipping huge 50996 token file...
Skipping huge 32700 token file...


KeyboardInterrupt: 